<a href="https://colab.research.google.com/github/krishhh-na/Tareas-FIS1344-1/blob/main/Tarea_11_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np                                            #importamos bibliotecas
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

np.random.seed(0)                                               #generamos datos
x_data = np.linspace(1, 10, 30)
y_true = (x_data/2)**-3 + x_data**2
y_data = y_true + np.random.normal(0, 0.2, size=x_data.shape)

x = jnp.array(x_data)                                          #convertimos los datos a arreglos JAX
y = jnp.array(y_data)

In [ ]:
def modelo(x, theta):                                      #definimos el modelo
    return (theta[0]*x)**2 + (theta[1]*x)**(-3)

def suma_cuadrados(theta, x, y):                           #definimos la función a minimizar
    predicciones = modelo(x, theta)
    return jnp.sum((y - predicciones)**2)

gradiente = jax.jit(jax.grad(suma_cuadrados))             #calcular gradiente
theta_inicial = jnp.array([0.8, 0.8])                     #elegimos esos valores para comparar ambos metodos en el mismo punto


In [ ]:
def descenso_simple(theta, x, y, learning_rate=3e-7, iteraciones=3000):       #método simple
    for i in range(iteraciones):                                             #repetir procedimiento
        g = gradiente(theta, x, y)
        theta = theta - learning_rate*g                                      #actualizamos theta
    return theta

theta_simple = descenso_simple(theta_inicial, x, y)                         #ejecutamos el métodp
print("Parámetros:", theta_simple)
print("Suma de cuadrados:", suma_cuadrados(theta_simple, x, y))

Parámetros: [1.0007713 0.6590425]
Suma de cuadrados: 32.437637


In [ ]:

def descenso_adam(theta, x, y, learning_rate=0.01, beta1=0.9, beta2=0.999, epsilon=1e-8, iteraciones=3000):       #método adams

    m = jnp.zeros_like(theta)                                                           #inicializamos
    v = jnp.zeros_like(theta)

    for i in range(1, iteraciones+1):                                                  #repetimos el procedimiento

        g = gradiente(theta, x, y)                                                     #gradiente

        m = beta1*m + (1-beta1)*g                                                      #actualizamos promedios
        v = beta2*v + (1-beta2)*(g**2)

        m_hat = m/(1-beta1**i)                                                         #corregimos promedios
        v_hat = v/(1-beta2**i)

        theta = theta - learning_rate*m_hat / (                                        #actualizamos theta
            jnp.sqrt(v_hat) + epsilon)

    return theta


theta_adam = descenso_adam(theta_inicial, x, y)                                        #ejectuamos adams
print("Parámetros:", theta_adam)
print("Suma de cuadrados:",
      suma_cuadrados(theta_adam, x, y))


Parámetros: [1.0004886  0.49177524]
Suma de cuadrados: 1.3524117


si comparamos los métodos, vemos que el método Adams (1.5) tuvo un mejor ajuste de descenso que el gradiente sencillo (32). Esto ocurre porque Adams utiliza información de gradientes anteriores, mientras que el método sencillo usa el mismo valor de learning_rate durante todas las iteraciones.